In [1]:
# Check GPU availability
!nvidia-smi

Mon Mar  2 17:09:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
# Install Ultralytics YOLO
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.7 MB/s eta 0:00:00


In [4]:
# Mount Google Drive to access dataset zip

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Unzip dataset into Colab workspace

!unzip "/content/drive/MyDrive/yolo_dataset.zip" -d /content/

Archive:  /content/drive/MyDrive/yolo_dataset.zip
   creating: /content/yolo_dataset/
  inflating: /content/yolo_dataset/data.yaml  
   creating: /content/yolo_dataset/images/
   creating: /content/yolo_dataset/images/test/
  inflating: /content/yolo_dataset/images/test/image_012.jpg  
  inflating: /content/yolo_dataset/images/test/image_027.jpg  
  inflating: /content/yolo_dataset/images/test/image_031.jpg  
  inflating: /content/yolo_dataset/images/test/image_068.jpg  
  inflating: /content/yolo_dataset/images/test/image_074.jpg  
  inflating: /content/yolo_dataset/images/test/image_079.jpg  
  inflating: /content/yolo_dataset/images/test/image_089.jpg  
  inflating: /content/yolo_dataset/images/test/image_092.jpg  
  inflating: /content/yolo_dataset/images/test/image_097.jpg  
  inflating: /content/yolo_dataset/images/test/image_100.jpg  
  inflating: /content/yolo_dataset/images/test/image_101.jpg  
  inflating: /content/yolo_dataset/images/test/image_113.jpg  
  inflating: /conten

### **Verify Structure**

In [6]:
# Check extracted folder

!ls /content/yolo_dataset
!ls /content/yolo_dataset/images
!ls /content/yolo_dataset/labels

data.yaml  images  labels
test  train  val
test  train  val


In [7]:
# data.yaml inside yolo_dataset

%%writefile /content/yolo_dataset/data.yaml

path: /content/yolo_dataset

train: images/train
val: images/val
test: images/test

nc: 1
names: ['pothole']

Overwriting /content/yolo_dataset/data.yaml


In [8]:
# Check number of images in each split

!ls /content/yolo_dataset/images/train | wc -l
!ls /content/yolo_dataset/images/val | wc -l
!ls /content/yolo_dataset/images/test | wc -l

1509
431
217


## **Start Training**

In [9]:
# Import YOLO
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Start training
model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    workers=2,
    device=0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, in

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d23bceb7080>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

### **Evaluate on Test Set**

In [10]:
# Load best trained model

from ultralytics import YOLO

model = YOLO('/content/runs/detect/train/weights/best.pt')

# Evaluate on test set
metrics = model.val(
    data='/content/yolo_dataset/data.yaml',
    split='test'
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 56.7±54.2 MB/s, size: 419.8 KB)
val: Scanning /content/yolo_dataset/labels/test... 217 images, 73 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 217/217 349.1it/s 0.6s
val: /content/yolo_dataset/images/test/img-556.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.0263      1.0525       1.075]
val: New cache created: /content/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 3.0it/s 4.6s
                   all        216        370      0.943       0.85      0.906      0.703
Speed: 2.1ms preprocess, 9.1ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/val


### **Load Best Model**

In [15]:
from ultralytics import YOLO

# Load best trained weights

model = YOLO('/content/runs/detect/train/weights/best.pt')

### **Run Prediction**

In [16]:
model.predict(
    source='/content/yolo_dataset/images/test',
    save=True,
    project='/content/pothole_results',
    name='test_predictions',
    conf=0.25,
    device=0
)


image 1/217 /content/yolo_dataset/images/test/image_012.jpg: 448x640 1 pothole, 18.5ms
image 2/217 /content/yolo_dataset/images/test/image_027.jpg: 192x640 (no detections), 9.3ms
image 3/217 /content/yolo_dataset/images/test/image_031.jpg: 640x640 (no detections), 17.3ms
image 4/217 /content/yolo_dataset/images/test/image_068.jpg: 448x640 (no detections), 11.9ms
image 5/217 /content/yolo_dataset/images/test/image_074.jpg: 288x640 (no detections), 8.9ms
image 6/217 /content/yolo_dataset/images/test/image_079.jpg: 448x640 (no detections), 7.9ms
image 7/217 /content/yolo_dataset/images/test/image_089.jpg: 384x640 (no detections), 7.7ms
image 8/217 /content/yolo_dataset/images/test/image_092.jpg: 448x640 (no detections), 7.7ms
image 9/217 /content/yolo_dataset/images/test/image_097.jpg: 640x448 (no detections), 7.8ms
image 10/217 /content/yolo_dataset/images/test/image_100.jpg: 224x640 (no detections), 7.8ms
image 11/217 /content/yolo_dataset/images/test/image_101.jpg: 480x640 (no detecti

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'pothole'}
 obb: None
 orig_img: array([[[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [57, 61, 61],
         [60, 65, 64],
         [55, 60, 59]],
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [60, 65, 64],
         [61, 66, 65],
         [55, 60, 59]],
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [63, 68, 68],
         [55, 60, 59],
         [60, 65, 64]],
 
        ...,
 
        [[13, 12, 14],
         [ 2,  2,  4],
         [ 8,  8,  8],
         ...,
         [38, 42, 44],
         [34, 38, 40],
         [43, 47, 48]],
 
        [[ 6,  5,  7],
         [ 8,  7, 10],
         [ 3,  3,  3],
         ...,
         [45, 49, 51],
         [44, 48, 49],
         [44, 48, 49]],
 
        [[ 5,  4,  6],
   

### **Verify Saved Images**

In [17]:
!ls /content/pothole_results/test_predictions

image_012.jpg  image_476.jpg  img-23.jpg   img-585.jpg	    potholes355.jpg
image_027.jpg  image_488.jpg  img-244.jpg  img-588.jpg	    potholes356.jpg
image_031.jpg  image_498.jpg  img-257.jpg  img-609.jpg	    potholes362.jpg
image_068.jpg  image_500.jpg  img-259.jpg  img-613.jpg	    potholes376.jpg
image_074.jpg  image_508.jpg  img-268.jpg  img-626.jpg	    potholes389.jpg
image_079.jpg  image_512.jpg  img-272.jpg  img-65.jpg	    potholes38.jpg
image_089.jpg  image_521.jpg  img-295.jpg  img-663.jpg	    potholes406.jpg
image_092.jpg  image_536.jpg  img-305.jpg  img-666.jpg	    potholes421.jpg
image_097.jpg  image_558.jpg  img-321.jpg  img-672.jpg	    potholes425.jpg
image_100.jpg  image_571.jpg  img-330.jpg  img-674.jpg	    potholes428.jpg
image_101.jpg  image_572.jpg  img-333.jpg  img-688.jpg	    potholes429.jpg
image_113.jpg  image_609.jpg  img-335.jpg  img-697.jpg	    potholes430.jpg
image_117.jpg  image_614.jpg  img-343.jpg  img-69.jpg	    potholes434.jpg
image_135.jpg  image_625.jpg

In [18]:
model.predict(
    source='/content/yolo_dataset/images/test',
    save=True,
    project='/content/pothole_results',
    name='test_predictions_lowconf',
    conf=0.10,
    device=0
)


image 1/217 /content/yolo_dataset/images/test/image_012.jpg: 448x640 1 pothole, 11.1ms
image 2/217 /content/yolo_dataset/images/test/image_027.jpg: 192x640 (no detections), 26.1ms
image 3/217 /content/yolo_dataset/images/test/image_031.jpg: 640x640 (no detections), 21.3ms
image 4/217 /content/yolo_dataset/images/test/image_068.jpg: 448x640 (no detections), 60.2ms
image 5/217 /content/yolo_dataset/images/test/image_074.jpg: 288x640 (no detections), 7.0ms
image 6/217 /content/yolo_dataset/images/test/image_079.jpg: 448x640 (no detections), 6.7ms
image 7/217 /content/yolo_dataset/images/test/image_089.jpg: 384x640 (no detections), 6.4ms
image 8/217 /content/yolo_dataset/images/test/image_092.jpg: 448x640 (no detections), 6.4ms
image 9/217 /content/yolo_dataset/images/test/image_097.jpg: 640x448 (no detections), 6.3ms
image 10/217 /content/yolo_dataset/images/test/image_100.jpg: 224x640 (no detections), 6.2ms
image 11/217 /content/yolo_dataset/images/test/image_101.jpg: 480x640 (no detect

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'pothole'}
 obb: None
 orig_img: array([[[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [57, 61, 61],
         [60, 65, 64],
         [55, 60, 59]],
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [60, 65, 64],
         [61, 66, 65],
         [55, 60, 59]],
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [63, 68, 68],
         [55, 60, 59],
         [60, 65, 64]],
 
        ...,
 
        [[13, 12, 14],
         [ 2,  2,  4],
         [ 8,  8,  8],
         ...,
         [38, 42, 44],
         [34, 38, 40],
         [43, 47, 48]],
 
        [[ 6,  5,  7],
         [ 8,  7, 10],
         [ 3,  3,  3],
         ...,
         [45, 49, 51],
         [44, 48, 49],
         [44, 48, 49]],
 
        [[ 5,  4,  6],
   

In [19]:
model.val()

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2273.5±1960.4 MB/s, size: 537.6 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 431 images, 159 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 431/431 54.8Mit/s 0.0s
val: /content/yolo_dataset/images/val/image_499.jpg: corrupt JPEG restored and saved
val: /content/yolo_dataset/images/val/img-693.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1295      1.0935]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 3.8it/s 7.2s
                   all        430        764      0.917      0.861      0.924      0.716
Speed: 1.3ms preprocess, 4.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/runs/detect/val2


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d23b8a50d10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [20]:
# Check training folder

!ls /content/runs/detect/train

args.yaml			 results.csv	      val_batch0_labels.jpg
BoxF1_curve.png			 results.png	      val_batch0_pred.jpg
BoxP_curve.png			 train_batch0.jpg     val_batch1_labels.jpg
BoxPR_curve.png			 train_batch1.jpg     val_batch1_pred.jpg
BoxR_curve.png			 train_batch2.jpg     val_batch2_labels.jpg
confusion_matrix_normalized.png  train_batch8550.jpg  val_batch2_pred.jpg
confusion_matrix.png		 train_batch8551.jpg  weights
labels.jpg			 train_batch8552.jpg


In [22]:
# best.pt location

!ls /content/runs/detect/train/weights

best.pt  last.pt


In [23]:
# results.png Location

!ls /content/runs/detect/train

args.yaml			 results.csv	      val_batch0_labels.jpg
BoxF1_curve.png			 results.png	      val_batch0_pred.jpg
BoxP_curve.png			 train_batch0.jpg     val_batch1_labels.jpg
BoxPR_curve.png			 train_batch1.jpg     val_batch1_pred.jpg
BoxR_curve.png			 train_batch2.jpg     val_batch2_labels.jpg
confusion_matrix_normalized.png  train_batch8550.jpg  val_batch2_pred.jpg
confusion_matrix.png		 train_batch8551.jpg  weights
labels.jpg			 train_batch8552.jpg


In [24]:
# Confusion Metrics

model.val()

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1247.7±637.0 MB/s, size: 413.2 KB)
val: Scanning /content/yolo_dataset/labels/val.cache... 431 images, 159 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 431/431 113.0Mit/s 0.0s
val: /content/yolo_dataset/images/val/image_499.jpg: corrupt JPEG restored and saved
val: /content/yolo_dataset/images/val/img-693.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [     1.1295      1.0935]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 4.2it/s 6.4s
                   all        430        764      0.917      0.861      0.924      0.716
Speed: 1.8ms preprocess, 3.3ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/runs/detect/val3


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d23b7a3f920>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [25]:
# Training Log

!ls /content/runs/detect/train | grep csv

results.csv


In [26]:
# Save Everything to Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# Copying everything to drive

!cp -r /content/runs/detect/train /content/drive/MyDrive/pothole_project_backup